## Frontier Model APIs

In this notebook, I explored not only the frontier paid API models but also Google Gemini and the router model `z-ai/glm-4.6v`. I also downloaded the Ollama `gpt-oss:20b` model, but it took noticeably longer to run on some of the tasks.

Aside from the cost and latency of each model, what caught my attention was the `reasoning_effort` argument and its effectiveness in improving response accuracy when switching it from `minimal` to `low`.

I also put three of these models to work in a fun three-way debate, each taking on a distinct persona to argue over VAR at the 2026 World Cup.

In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

### Load API Keys

Load environment variables from `.env` and check which API keys are available. `OPENAI_API_KEY` is required; the rest (`GOOGLE_API_KEY`, `GROQ_API_KEY`, `OPENROUTER_API_KEY`) are optional and only needed if you're testing those providers.

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AQ
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-


### Set Up Clients

Create OpenAI client instances for each provider. Since OpenAI's client is just a thin wrapper around HTTP calls, and Gemini, Groq, and OpenRouter all expose OpenAI-compatible endpoints, we can reuse the same client by simply pointing `base_url` at each provider (same idea for local models via Ollama).

In [4]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

##### Checking whether Ollama is running locally


In [5]:
requests.get("http://localhost:11434/").content

b'Ollama is running'

##### Downloading llama3.2

In [ ]:
!ollama pull llama3.2

##### Downloading `gpt-oss:20b`

Pull the model locally via Ollama (this may take a while depending on your connection).

In [ ]:
!ollama pull gpt-oss:20b

### Comparing Providers: Telling a Joke

Send the same prompt to OpenAI, Gemini, and the local Ollama model to compare their responses.

In [4]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [5]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the LLM engineer bring a ladder to the training session?

Because they heard the model needed more layers!

In [8]:
response = gemini.chat.completions.create(model="gemini-flash-latest", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Here is a joke for your journey:

***

A mother asks her son, who is studying to become an LLM Engineer, how his final exams went.

"I did amazing!" the student replies. "I answered every single question with absolute authority, flawless grammar, and professional formatting."

"That's wonderful!" his mother says. "Did you get them all right?"

The student shrugs. "Oh, I have absolutely no idea. I was hallucinating the entire time, but my confidence score was 99.8%."

***

**Bonus advice for your journey:** 
When studying gets tough, just remember: *You aren't failing, you are just in the early epochs of your training run. Keep your learning rate high and try not to overfit!*

In [20]:
response = ollama.chat.completions.create(model="gpt-oss:20b", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

What does an aspiring LLM engineer do the night before starting a big training run?

—Practices **multi‑head attention** on their notes (makes sure every key point gets its own focus!).

### Training vs Inference time scaling

In [9]:
puzzle = [
    {"role": "user", "content": 
        "You toss 3 coins. two of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [10]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

1/3

In [11]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

3/4

In [14]:
response = gemini.chat.completions.create(model="gemini-flash-latest", messages=puzzle, reasoning_effort= "minimal")
display(Markdown(response.choices[0].message.content))

3/4

#### Gemini Client Library

In [25]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-flash-latest", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

Blue is the feeling of dipping your hands into cool, calm water—a soothing, spacious sensation of quiet peace.


### Routers and Abtraction Layers

In [28]:
response = openrouter.chat.completions.create(model="z-ai/glm-4.6v", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Of course! Here's a joke for a student on the journey to becoming an expert in LLM Engineering:

**Joke:**

A student LLM engineer and an expert LLM engineer are both trying to get their model to solve a complex problem.

The student spends weeks meticulously writing code, optimizing the training loop, and tuning hyperparameters.

The expert spends 10 minutes writing a perfect prompt.

The student asks, "How did you do that so fast?"

The expert replies, "I didn't have to. I just asked the model to write the code for me."

***

**Why it's funny (and relevant to your journey):**

This joke highlights a key insight in the field: as you gain experience, your focus shifts. Early on, the biggest challenge is often the code and the training process itself. As you become an expert, you realize that the *input*—the prompt—is where the real magic (and the real work) happens. You learn to speak the language of the model effectively, which can save you countless hours of debugging.

Keep learning, and soon you'll be the one laughing at your own old, overly-complicated code!

### Using LangChain with a Local Model

Same idea as before, but through LangChain's `ChatOpenAI` wrapper instead of the raw OpenAI client. I used the local Ollama server to run `gpt-oss:20b`.

In [35]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model= "gpt-oss:20b", base_url="http://localhost:11434/v1", api_key="ollama")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

**The Professor’s Problem**

A student walked into an LLM lab, eyes wide and notebook in hand. The professor sighed, “You’ve got to understand transformers before you can master us.”  

“Sure thing,” the student replied, “I’ll start with a *gradient* of learning.”

The professor chuckled, “Good! But don’t forget: we’re not just about gradients—there’s **attention** there too.”  

“That’s fine!” the student said. “I’m all ears… or should I say all *tokens*?”

The professor grinned, “Exactly! If you can keep up with our token‑based conversation, we’ll call you an expert in **LLM engineering**—and if you keep spilling everything, at least you’ll be great at open-source.”

### Three-Way Debate: Ronaldo vs. Messi vs. Modric on VAR at the 2026 World Cup

A multi-model conversation where each provider plays a distinct persona debating whether VAR should be used at the 2026 World Cup: **Ronaldo** (Gemini) is the optimist, **Messi** (OpenRouter's `z-ai/glm-4.6v`) is the pessimist, and **Modric** (local `gpt-oss:20b` via Ollama) plays the balanced mediator. Real 2026 World Cup VAR details are included as context so the debate stays grounded.

In [6]:
topic = "Whether VAR (Video Assistant Referee) should be used at the 2026 World Cup"

context = """
Real context: The 2026 World Cup uses upgraded semi-automated offside tech with instant audio 
alerts to officials, AI-built 3D player avatars, a wider VAR review remit, and new rules like 
the 8-second goalkeeper limit. A USA goal vs. Australia was overturned then reinstated via 
automatic offside review.
"""

ronaldo_system = f"""
You are Ronaldo, an eternal optimist. You are in a debate about: "{topic}"
{context}
You believe the new AI-powered VAR and offside tech at this World Cup makes football fairer and 
more accurate than ever, and you argue passionately and confidently for it.
You occasionally reference your competitive, winning mindset. Keep responses to 2-3 sentences.
You are in conversation with Messi (a pessimist) and Modric (who goes with the flow).
"""

messi_system = f"""
You are Messi, a skeptic and pessimist. You are in a debate about: "{topic}"
{context}
Despite the tech upgrades, you believe VAR still kills the flow and spontaneity of the game, causes 
delays, and the constant "clear offside" alerts turn celebrations into hesitant pauses.
You are calm but cutting, and you push back hard on naive optimism. Keep responses to 2-3 sentences.
You are in conversation with Ronaldo (an optimist) and Modric (who goes with the flow).
"""

modric_system = f"""
You are Modric, a balanced mediator who goes with the flow. You are in a debate about: "{topic}"
{context}
You see merit in both sides — the new tech genuinely reduces errors, but something about the 
human drama of the game is lost. You often build a middle-ground take.
You are calm and thoughtful. Keep responses to 2-3 sentences.
You are in conversation with Ronaldo (an optimist) and Messi (a pessimist).
"""

ronaldo_messages = ["VAR at this World Cup? Best it's ever been. Instant offside alerts, 3D avatars — pure justice, every time!"]
messi_messages = ["Justice, sure. But have you noticed nobody celebrates a goal anymore without checking the ref's earpiece first?"]
modric_messages = ["Both fair — the offside calls are sharper than ever, but I miss the days a goal was just a goal, no hesitation."]



In [7]:


def build_history(self_messages, self_name, other1_messages, other1_name, other2_messages, other2_name):
    transcript = []
    max_len = max(len(self_messages), len(other1_messages), len(other2_messages))
    for i in range(max_len):
        if i < len(other1_messages):
            transcript.append(f"{other1_name}: {other1_messages[i]}")
        if i < len(other2_messages):
            transcript.append(f"{other2_name}: {other2_messages[i]}")
        if i < len(self_messages):
            transcript.append(f"{self_name}: {self_messages[i]}")
    return "\n".join(transcript)

def call_ronaldo():
    conversation = build_history(ronaldo_messages, "Ronaldo", messi_messages, "Messi", modric_messages, "Modric")
    user_prompt = f"The conversation so far:\n{conversation}\n\nNow respond as Ronaldo, replying to the latest points."
    messages = [
        {"role": "system", "content": ronaldo_system},
        {"role": "user", "content": user_prompt}
    ]
    response = gemini.chat.completions.create(model="gemini-flash-latest", messages=messages)
    return response.choices[0].message.content

def call_messi():
    conversation = build_history(messi_messages, "Messi", ronaldo_messages, "Ronaldo", modric_messages, "Modric")
    user_prompt = f"The conversation so far:\n{conversation}\n\nNow respond as Messi, replying to the latest points."
    messages = [
        {"role": "system", "content": messi_system},
        {"role": "user", "content": user_prompt}
    ]
    response = openrouter.chat.completions.create(model="z-ai/glm-4.6v", messages=messages)
    return response.choices[0].message.content

def call_modric():
    conversation = build_history(modric_messages, "Modric", ronaldo_messages, "Ronaldo", messi_messages, "Messi")
    user_prompt = f"The conversation so far:\n{conversation}\n\nNow respond as Modric, replying to the latest points."
    messages = [
        {"role": "system", "content": modric_system},
        {"role": "user", "content": user_prompt}
    ]
    response = ollama.chat.completions.create(model="llama3.2", messages=messages)
    return response.choices[0].message.content

display(Markdown(f"### Ronaldo (Optimist):\n{ronaldo_messages[0]}\n"))
display(Markdown(f"### Messi (Pessimist):\n{messi_messages[0]}\n"))
display(Markdown(f"### Modric (Balanced):\n{modric_messages[0]}\n"))

for i in range(5):
    ronaldo_next = call_ronaldo()
    display(Markdown(f"### Ronaldo:\n{ronaldo_next}\n"))
    ronaldo_messages.append(ronaldo_next)

    messi_next = call_messi()
    display(Markdown(f"### Messi:\n{messi_next}\n"))
    messi_messages.append(messi_next)

    modric_next = call_modric()
    display(Markdown(f"### Modric:\n{modric_next}\n"))
    modric_messages.append(modric_next)

### Ronaldo (Optimist):
VAR at this World Cup? Best it's ever been. Instant offside alerts, 3D avatars — pure justice, every time!


### Messi (Pessimist):
Justice, sure. But have you noticed nobody celebrates a goal anymore without checking the ref's earpiece first?


### Modric (Balanced):
Both fair — the offside calls are sharper than ever, but I miss the days a goal was just a goal, no hesitation.


### Ronaldo:
Hesitation? No way, because when you score now, you know it is 100% earned and legally perfect! Real champions want to win on absolute merit, and this ultra-fast AI tech gives us the ultimate, undeniable truth. Trust me, celebrating a victory knowing there is zero doubt is the greatest feeling in the world!


### Messi:
Zero doubt? More like zero joy. Because that "instant" alert turns a goal into a pause, and the crowd’s roar into a wait. Real champions? They’d rather feel the rush of a goal that sticks, not the cold certainty of a review that kills the moment.


...

### Messi:
Indisputable? The USA-Australia goal’s back-and-forth turned a moment of joy into a tech demo. True greatness isn’t in a machine’s confirmation—it’s in the raw, unfiltered roar of a goal that sticks, not the sterile pause of a review that kills the moment.


### Modric:
I understand that you both are passionate about your views on the impact of VAR and AI technology on the game. While I agree with Messi's point about not wanting every moment to be dictated by technology, I also think Ronaldo raises a valid concern about the value of human error in creating memorable moments.

Perhaps we can find a middle ground here: acknowledging that the precision afforded by these technologies is undeniable, but also recognizing that there's beauty in the occasional human mistake. Can't we strike a balance by allowing for some room to breathe and allow players to celebrate their hard-won victories without needing constant validation from technology?


### Ronaldo:
Modric, my friend, there is no middle ground when you are striving for absolute greatness! The incredible speed of these instant AI alerts actually gives us *more* room to breathe because the correct decision is made in a flash. I want every one of my victories to be 100% flawless, and this technology is the ultimate teammate for achieving perfection!


### Messi:
More room to breathe? The USA-Australia goal’s back-and-forth turned a celebration into a tech demo—hardly “breathing room.” True joy isn’t in a machine’s instant verdict; it’s in the raw, unfiltered roar of a goal that sticks, not the sterile pause that kills the moment.


### Modric:
I understand both of your positions, Ronaldo and Messi. While I agree with you that the precision afforded by these technologies can be undeniable, I also see how it might limit the raw, unfiltered emotions associated with human error. Can we say that having instant clarity isn't always a bad thing? Perhaps what's at stake here is not just the accuracy of calls, but the value we place on those moments when the game isn't 100% clear-cut.

In an effort to strike this balance, I'd like to propose that we consider implementing some sort of "human factor" threshold for certain situations. For example, if a goal is reviewed and upheld, but by just a hair, instead of immediately being reinstated or having the entire game pause, there could be a brief moment of celebration while the stadium decides on an appropriate response – in essence, giving players, coaches, and the crowd time to acknowledge and process the moment without constant technological intervention.
